In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from transformers import set_seed
import torch
from time import time
import evaluate
import numpy as np
import collections

In [ ]:
set_seed(42)
torch.manual_seed(42)
batch_size = 32
epochs = 4
lr=5e-5
shuffle = True
clean_text = True
prompt_tokens = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
selection_criterion = 'eval_loss' # Choose between 'eval_matthews_correlation' and 'eval_loss'

In [ ]:
squad_dataset = load_dataset("squad")
squad_dataset['test'] =squad_dataset['validation']
del squad_dataset['validation']
train_val_split = squad_dataset['train'].train_test_split(test_size=0.1, seed=42)
squad_dataset['train'] = train_val_split['train']
squad_dataset['val'] = train_val_split['test']

In [ ]:
def add_end_idx(example):
    gold_text = example['answers']['text'][0]
    start_idx = example['answers']['answer_start'][0]
    end_idx = start_idx + len(gold_text)

    # sometimes squad answers are off by a character or two – fix this
    if example['context'][start_idx:end_idx] == gold_text:
        pass
        # example['answers']['answer_end'] = [end_idx]
    elif example['context'][start_idx-1:end_idx-1] == gold_text:
        example['answers']['answer_start'] = [start_idx - 1]
        # example['answers']['answer_end'] = [end_idx - 1]  # When the gold label is off by one character
    elif example['context'][start_idx-2:end_idx-2] == gold_text:
        example['answers']['answer_start'] = [start_idx - 2]
        # example['answers']['answer_end'] = [end_idx - 2]  # When the gold label is off by two characters
    else:
        example['answers']['answer_start'] = [start_idx]
        # example['answers']['answer_end'] = [end_idx]
    return example


In [ ]:
squad_dataset = squad_dataset.map(add_end_idx)

In [ ]:
from peft import PromptTuningConfig, get_peft_model, TaskType

In [ ]:
peft_config = PromptTuningConfig(
    task_type=TaskType.QUESTION_ANS,
    prompt_tuning_init="RANDOM",
    num_virtual_tokens=prompt_tokens,
    tokenizer_name_or_path="distilbert-base-uncased",
    num_layers=6,
    num_attention_heads=12,
    token_dim=768,
)
peft_config.modules_to_save = None

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased")

In [ ]:
model = get_peft_model(model, peft_config = peft_config)
model.print_trainable_parameters()
model = model.to(device)

In [ ]:
max_length = 384
stride = 128

In [ ]:
def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char_idx = answer["answer_start"][0]
        end_char_idx = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char_idx or offset[context_end][1] < end_char_idx:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char_idx:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char_idx:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
data_train,data_val,data_test = squad_dataset['train'],squad_dataset['val'],squad_dataset['test']

In [ ]:
train_tokenized = data_train.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=data_train.column_names,
)
val_tokenized = data_val.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=data_val.column_names,
)

In [ ]:
def preprocess_test_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [ ]:
test_tokenized = data_test.map(
    preprocess_test_examples,
    batched=True,
    remove_columns=data_test.column_names,
)

In [ ]:
metric = evaluate.load("squad")
n_best = 20
max_answer_length = 30

In [ ]:
def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in (examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all features associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [ ]:
results_path = './results/batch_size_{}_epochs_{}_lr_{}'.format(batch_size, epochs, lr)
log_path = './logs/batch_size_{}_epochs_{}_lr_{}'.format(batch_size, epochs, lr)

In [ ]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir=results_path,
    num_train_epochs=epochs, 
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=256,
    learning_rate=lr,
    logging_dir=log_path,
    logging_steps=100,
    evaluation_strategy='steps',
    save_steps=100,
    eval_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='tensorboard',
    seed=42,
    run_name='Batch size: {}, Epochs: {}, LR: {}'.format(batch_size, epochs, lr),
    lr_scheduler_type='constant',
    warmup_steps=0,
    fp16=True,
    remove_unused_columns=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
)

In [ ]:
start_time = time()

In [ ]:
trainer.train()

In [ ]:
# Save the best model
'''
Keep a local copy of the best model
'''
best_model_path = "./best_model_{batch_size}_{epochs}_{lr}".format(batch_size=batch_size, epochs=epochs, lr=lr)
trainer.model.save_pretrained(best_model_path)

In [ ]:
with open ('./time.txt', 'a+') as f:
    f.write('Batch size: {}, Epochs: {}, LR: {} - Training time: {:.2f} seconds\n'.format(batch_size, epochs, lr, time()-start_time))

In [ ]:
predictions, _, _ = trainer.predict(test_dataset=test_tokenized.select(range(3)))
start_logits, end_logits = predictions

In [ ]:
compute_metrics(start_logits, end_logits, test_tokenized, data_test)